# Overview

Moving on to lab 2, we'll cover 3 Topics:

1. Manual Real Time Model Deployment (lab2a_notebook_realtime_manual_deployment.ipynb)
2. Automated Real Time Model Deployment with Manual Approval (lab2b_notebook_realtime_automated_deployment.ipynb)
3. Batch Model Deployment (lab_2c_notebook_batch_deployment.ipynb)

In this notebook, we revisit how to register models, best practice in setting model alias and what to check in staging before moving to production. We will use the materials that we have gone through in this notebook and automate it in notebook `lab2b_notebook_realtime_automated_deployment.ipynb`


# Revisiting How to Deploy / Register model to Unity Catalog

In [0]:
import mlflow
# Step 1 : Set the regsitry URI to databricks-uc (Always databricks-uc, representing Databrick's managed Unity Catalog)
mlflow.set_registry_uri("databricks-uc") 

# Step 2: Define the catalog, schema and your model name
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model" 

# Step 3: Select the run your want to register & point to the model artifact path
run_id = "fdc1d4409e974ae4a3b7fb7098b5479c" 
model_uri = f"runs:/{run_id}/model-artifacts"

# Step 4: Register your model to Unity Catalog
registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

# 1. Manual Deployment and Approval Workflow

Remember we have 3 deployment stage:
1. Dev
2. Staging
3. Production

All the work you have done (Feature engineering, training, evaluation) including registering a model to the unity catalog sits on the Dev stage. You may register more than 1 model to Unity Catalog if you so choose. Just remember to give a comment to the model to remember why you register the model and also the current model you are currently developing. However, try to only deploy model the best model for your experiment runs to avoid cluttering your registered models

In [0]:
# How to give your registered model a comment

from mlflow.tracking.client import MlflowClient

client = MlflowClient()

client.update_model_version(
    name = registered_model.name,
    version = registered_model.version,
    description = "V0.0.1 - Simple Feature Engineering. Highest AUC", # We're currently developing model version v0.0.1
)

In [0]:
# Set the chosen model alias as Dev to signify that this is the final model you want to deploy
client.set_registered_model_alias(
    name = model_name,
    alias = "dev",
    version = "3"
)

# Moving to Staging Environment

## Real Time Deployment

In this step, we'll take the model that we have aliased as Dev and serve it

In [0]:
import mlflow
from mlflow.tracking.client import MlflowClient

mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model" 
registered_model = client.get_model_version_by_alias(
    name=model_name,
    alias="dev"
)

# Now 'registered_model' holds the ModelVersion object
print(registered_model.name)
print(registered_model.version)
print(registered_model.description)

Next, serve the model in staging

In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
endpoint_name = f"example-titanic-serving-stg"

endpoint_config = {
    "served_entities": [
        {
            "name": "titanic-model-v0_0_1",                             # Give this specific served model a reference name within the endpoint
            "entity_name": f"{registered_model.name}",                  # The exact path to your model in Unity Catalog
            "entity_version": f"{registered_model.version}",            # Must be passed as a string
            "workload_size": "Small",                                   # Small, Medium, Large
            "scale_to_zero_enabled": True                               # True or False
        }
    ],
    "traffic_config" : {
        "routes" : [
            {
                "served_model_name" : "titanic-model-v0_0_1",
                "traffic_percentage": 100
            }
            
        ]
    }
}


# Safe code practice to avoid serving with the same endpoint
try:
    # Attempt to get the endpoint
    existing_endpoint = client.get_endpoint(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists.")
except Exception as e:
    # If not found, create the endpoint
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"Creating a new endpoint: {endpoint_name}")
        endpoint = client.create_endpoint(
            name=endpoint_name,
            config=endpoint_config
        )
    else:
        print(f"An error occurred: {e}")

Wait a couple of minute. Your model is updating to be served...

Next, let's do some testing in staging to make sure the result is expected and same as what you would get if you try to predict using the model offline

In [0]:
# Setup - Load your data
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
TRAIN_TABLE_NAME = "train"


train = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TRAIN_TABLE_NAME}").toPandas()
features = ["Fare", "Age", "Pclass", "SibSp", "Parch", "Sex"] 
X = train[features]
y = train["Survived"]

# Let's only test using 5 samples
X = X.head()
y = y.head()

In [0]:
X

In [0]:
y

In [0]:
# Setup - Load the model you deployed 
run_id = "fdc1d4409e974ae4a3b7fb7098b5479c" # this is the same run_id as the one you registered above
model_uri = f"runs:/{run_id}/model-artifacts"
loaded_model_pipeline = mlflow.sklearn.load_model(model_uri)
y_pred = loaded_model_pipeline.predict(X)


In [0]:
y_pred

Now, try to hit your model with the same features to see if the prediction are the same. But first, we need to create an access token first to be able to hit the model endpoint

- Step 1: Create a DATABRICKS_TOKEN
    - 1.1. Go to your profile (top-right) -> settings -> developer -> access tokens -> manage
    - 1.2. Generate new token and set API scope as "model serving" & "model serving inference"
    - 1.3. Copy token
- Step 2: Store your token in Databricks Secrets
    - 2.1. Open Databricks CLI (Bottom-right)
    - 2.2. Run `databricks secrets create-scope my-secrets`
    - 2.3. Run `databricks secrete put-secrete my-secrets databricks-token`
    - 2.4. You will be prompted to enter your secret 



In [0]:
import os
token = dbutils.secrets.get(scope='my-secrets', key='databricks-token')

In [0]:
import json
import requests

# Now to hit your model and get its prediction
url = 'https://dbc-a1a80dee-38ec.cloud.databricks.com/serving-endpoints/example-titanic-serving-stg/invocations' 
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

"""
The payload must be in the format of
{
    'dataframe_split': {
        'index': [0, 1, 2, 3, 4],
        'columns': ['Fare', 'Age', 'Pclass', 'SibSp', 'Parch', 'Sex'],
        'data': [
            [7.25, 22.0, 3, 1, 0, 'male'],
            [71.2833, 38.0, 1, 1, 0, 'female'],
            [7.925, 26.0, 3, 0, 0, 'female'],
            [53.1, 35.0, 1, 1, 0, 'female'],
            [8.05, 35.0, 3, 0, 0, 'male']
        ]
    }
}
"""
payload = {"dataframe_split": X.to_dict(orient="split")}
payload_json = json.dumps(payload, allow_nan=True)
response = requests.request(method='POST', headers=headers, url=url, data=payload_json)

if response.status_code != 200:
    raise Exception(f'Request failed with status {response.status_code}, {response.text}')

print(f"Offline Prediction: {y_pred}")
print(f"Online Prediction: {response.json()}")

Predictions are the same, we have confirmed that the correct model is deployed succesfully. Next, let's do a load test to make sure that the latency and throughput requirement is fulfilled. Use the boilerplate code below to perform a load test (You can copy and reuse it)

In [0]:
X.head(1)

In [0]:
import json
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np

# Configuration
url = 'https://dbc-a1a80dee-38ec.cloud.databricks.com/serving-endpoints/example-titanic-serving-stg/invocations' # Future Use: Modify this
token = dbutils.secrets.get(scope='my-secrets', key='databricks-token') 
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# Modify X slightly, in production serving, a single request will only contain a single sample. Thus, we use head(1) to simulate a single request to predict for a single sample
payload = {"dataframe_split": X.head(1).to_dict(orient="split")}
payload_json = json.dumps(payload, allow_nan=True)

In [0]:
def send_request(request_id):
    """Send a single request and record metrics"""
    start_time = time.time()
    try:
        response = requests.post(url, headers=headers, data=payload_json, timeout=30)
        latency = time.time() - start_time
        
        return {
            'request_id': request_id,
            'status_code': response.status_code,
            'latency': latency,
            'success': response.status_code == 200,
            'response': response.json() if response.status_code == 200 else None,
            'error': None
        }
    except Exception as e:
        latency = time.time() - start_time
        return {
            'request_id': request_id,
            'status_code': None,
            'latency': latency,
            'success': False,
            'response': None,
            'error': str(e)
        }

def load_test(num_requests=100, num_workers=10):
    """Run load test with concurrent requests"""
    print(f"Starting load test: {num_requests} requests with {num_workers} concurrent workers...\n")
    
    results = []
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(send_request, i) for i in range(num_requests)]
        
        for future in as_completed(futures):
            results.append(future.result())
            if len(results) % 1000 == 0:
                print(f"Completed {len(results)}/{num_requests} requests")
    
    total_time = time.time() - start_time
    
    # Calculate metrics
    latencies = [r['latency'] for r in results]
    successes = [r for r in results if r['success']]
    failures = [r for r in results if not r['success']]
    
    print("\n" + "="*50)
    print("LOAD TEST RESULTS")
    print("="*50)
    print(f"Total requests: {num_requests}")
    print(f"Concurrent workers: {num_workers}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Throughput: {num_requests/total_time:.2f} requests/second")
    print(f"\nSuccess rate: {len(successes)/num_requests*100:.2f}% ({len(successes)}/{num_requests})")
    print(f"Failed requests: {len(failures)}")
    print(f"\nLatency statistics (seconds):")
    print(f"  Min: {np.min(latencies):.3f}")
    print(f"  Max: {np.max(latencies):.3f}")
    print(f"  Mean: {np.mean(latencies):.3f}")
    print(f"  Median: {np.median(latencies):.3f}")
    print(f"  P95: {np.percentile(latencies, 95):.3f}")
    print(f"  P99: {np.percentile(latencies, 99):.3f}")
    
    if failures:
        print(f"\nError samples:")
        for f in failures[:5]:
            print(f"  Request {f['request_id']}: {f['error'] or f'HTTP {f['status_code']}'}")
    
    return results

In [0]:
# Assume that your endpoint receives at max 25 concurrent requests at a time and p99 latency requirement is 250ms. Can your model meet these requirements?
results = load_test(num_requests=5000, num_workers=25)

No, with 25 concurrent request, our model p99 latency is 311ms means that some request will timed out in this case you can play around with `"workload_size": "Medium"` to see if larger resource can meet your expected demand 

Once, you have fulfilled all tests in staging update your model alias to signify that the model has been deployed to staging succesfully

In [0]:
from mlflow.tracking.client import MlflowClient

# Set the chosen model alias as Dev to signify that this is the final model you want to deploy
client = MlflowClient()
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model" 
client.set_registered_model_alias(
    name = model_name,
    alias = "staging",
    version = "3"
)


# Moving to Production Environment

In the manual workflow, be sure to have a launch review meeting with your respective manager to discuss and verify the development and staging process before moving on to deploying in production

In [0]:
import mlflow
from mlflow.tracking.client import MlflowClient

mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()

CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model" 
registered_model = client.get_model_version_by_alias(
    name=model_name,
    alias="staging"
)

# Now 'registered_model' holds the ModelVersion object
print(registered_model.name)
print(registered_model.version)
print(registered_model.description)

In [0]:
from mlflow.deployments import get_deploy_client
client = get_deploy_client("databricks")
endpoint_name = f"example-titanic-serving-prd"

endpoint_config = {
    "served_entities": [
        {
            "name": "titanic-model-v0_0_1",                             
            "entity_name": f"{registered_model.name}",                  
            "entity_version": f"{registered_model.version}",            
            "workload_size": "Small",                                   
            "scale_to_zero_enabled": True                              
        }
    ],
    "traffic_config" : {
        "routes" : [
            {
                "served_model_name" : "titanic-model-v0_0_1",
                "traffic_percentage": 100
            }
            
        ]
    }
}


# Safe code practice to avoid serving with the same endpoint
try:
    # Attempt to get the endpoint
    existing_endpoint = client.get_endpoint(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists.")
except Exception as e:
    # If not found, create the endpoint
    if "RESOURCE_DOES_NOT_EXIST" in str(e):
        print(f"Creating a new endpoint: {endpoint_name}")
        endpoint = client.create_endpoint(
            name=endpoint_name,
            config=endpoint_config
        )
    else:
        print(f"An error occurred: {e}")

In [0]:
from mlflow.tracking.client import MlflowClient

# Set the chosen model alias as Dev to signify that this is the final model you want to deploy
client = MlflowClient()
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.example_titanic_model" 
client.set_registered_model_alias(
    name = model_name,
    alias = "production",
    version = "3"
)